In [1]:
import llama_index

print("LlamaIndex version:", llama_index.__version__)
print("W7D3 notebook started")

ModuleNotFoundError: No module named 'llama_index'

In [2]:
import llama_index

print("LlamaIndex version:", llama_index.__version__)
print("W7D3 notebook started")

AttributeError: module 'llama_index' has no attribute '__version__'

In [3]:
import importlib.metadata
import llama_index

print("LlamaIndex version:", importlib.metadata.version("llama-index"))
print("W7D3 notebook started")

LlamaIndex version: 0.14.25
W7D3 notebook started


In [5]:
from llama_index.embeddings.ollama import OllamaEmbedding

embed_model = OllamaEmbedding(
    model_name="nomic-embed-text",
    base_url="http://localhost:11434"
)

embedding = embed_model.get_text_embedding("What is a PID controller?")

print("Embedding created successfully")
print("Embedding dimensions:", len(embedding))

Embedding created successfully
Embedding dimensions: 768


In [6]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    "w7d3_docs"
).load_data()

print("Documents loaded:", len(documents))

Documents loaded: 5


In [7]:
from llama_index.core import Settings

Settings.embed_model = embed_model

print("LlamaIndex embedding model configured")

LlamaIndex embedding model configured


In [8]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(documents)

print("VectorStoreIndex created successfully")

VectorStoreIndex created successfully


In [10]:
from llama_index.llms.ollama import Ollama

llm = Ollama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434",
    request_timeout=120.0
)

Settings.llm = llm

print("Ollama LLM configured successfully")

Ollama LLM configured successfully


In [11]:
query_engine = index.as_query_engine()

print("QueryEngine created successfully")

QueryEngine created successfully


In [12]:
response = query_engine.query("What is a PID controller?")

print(response)

A PID controller is a feedback control mechanism that calculates an error between a desired setpoint and a measured process value. It then uses proportional, integral, and derivative terms to reduce this error and improve the system's response.


In [13]:
queries = [
    "What are the main terms of a PID controller?",
    "What is ROS 2 used for?",
    "What communication methods does ROS 2 provide?",
    "What is computer vision used for in robotics?",
    "What can OpenCV provide?",
    "What sensors are commonly used in autonomous vehicles?",
    "What are the main components used by autonomous vehicles to navigate?",
    "What is machine learning used for in robotics?",
    "What is the difference between supervised and unsupervised learning?"
]

for i, query in enumerate(queries, start=2):
    response = query_engine.query(query)
    print(f"\nQuery {i}: {query}")
    print(f"Answer: {response}")


Query 2: What are the main terms of a PID controller?
Answer: The main terms of a PID controller are proportional, integral, and derivative.

Query 3: What is ROS 2 used for?
Answer: ROS 2 is used for building distributed robotic applications by providing communication between nodes through topics, services, and actions.

Query 4: What communication methods does ROS 2 provide?
Answer: ROS 2 provides communication between nodes using topics, services, and actions.

Query 5: What is computer vision used for in robotics?
Answer: Computer vision is used in robotics to enable robots to understand and process visual information from cameras, which helps them to identify and interact with their environment more effectively.

Query 6: What can OpenCV provide?
Answer: OpenCV can provide image processing, feature detection, object detection, and video analysis capabilities.

Query 7: What sensors are commonly used in autonomous vehicles?
Answer: Common sensors used in autonomous vehicles includ

In [14]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

chroma_client = chromadb.PersistentClient(path="./w7d3_chroma")

chroma_collection = chroma_client.get_or_create_collection(
    name="w7d3_documents"
)

vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

print("ChromaDB connected successfully")

ChromaDB connected successfully


In [15]:
chroma_index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context
)

print("ChromaDB VectorStoreIndex created successfully")

ChromaDB VectorStoreIndex created successfully


In [16]:
chroma_query_engine = chroma_index.as_query_engine()

print("ChromaDB QueryEngine created successfully")

ChromaDB QueryEngine created successfully


In [17]:
import time

chroma_results = []

for i, query in enumerate(queries, start=1):
    start_time = time.perf_counter()

    response = chroma_query_engine.query(query)

    latency = time.perf_counter() - start_time

    chroma_results.append({
        "query": query,
        "answer": str(response),
        "latency_seconds": latency
    })

    print(f"\nQuery {i}: {query}")
    print(f"Answer: {response}")
    print(f"Latency: {latency:.4f} seconds")


Query 1: What are the main terms of a PID controller?
Answer: The main terms of a PID controller are proportional, integral, and derivative.
Latency: 31.9751 seconds

Query 2: What is ROS 2 used for?
Answer: ROS 2 is used for building distributed robotic applications by providing a framework for nodes to communicate with each other using topics, services, and actions.
Latency: 13.2713 seconds

Query 3: What communication methods does ROS 2 provide?
Answer: ROS 2 provides communication between nodes using topics, services, and actions.
Latency: 5.2812 seconds

Query 4: What is computer vision used for in robotics?
Answer: Computer vision in robotics is used to enable robots to understand and process visual information from cameras, which can be applied in various tasks such as object recognition, motion analysis, and scene understanding, thereby enhancing the robot's ability to interact with its environment autonomously.
Latency: 13.8050 seconds

Query 5: What can OpenCV provide?
Answe

In [18]:
for i, result in enumerate(chroma_results, start=1):
    print(f"Query {i}: {result['latency_seconds']:.4f} seconds")

Query 1: 31.9751 seconds
Query 2: 13.2713 seconds
Query 3: 5.2812 seconds
Query 4: 13.8050 seconds
Query 5: 7.6182 seconds
Query 6: 10.8363 seconds
Query 7: 11.1386 seconds
Query 8: 16.2464 seconds
Query 9: 20.9139 seconds


In [20]:
import time

query_10 = "What is the difference between supervised and unsupervised learning?"

start_time = time.perf_counter()
response_10 = chroma_query_engine.query(query_10)
latency_10 = time.perf_counter() - start_time

print("Query 10:", query_10)
print("Answer:", response_10)
print(f"Latency: {latency_10:.4f} seconds")

Query 10: What is the difference between supervised and unsupervised learning?
Answer: Supervised learning involves training models using labeled data, where each data point is associated with a known output or label. This method helps the model understand the relationship between inputs and outputs. In contrast, unsupervised learning uses unlabeled data to discover hidden patterns or intrinsic structures within the data.
Latency: 14.8435 seconds


In [21]:
chroma_latencies = [
    31.9751,
    13.2713,
    5.2812,
    13.8050,
    7.6182,
    10.8363,
    11.1386,
    16.2464,
    20.9139,
    14.8435
]

average_chroma_latency = sum(chroma_latencies) / len(chroma_latencies)

print(f"Average ChromaDB latency: {average_chroma_latency:.4f} seconds")

Average ChromaDB latency: 14.5930 seconds


In [22]:
import time

original_latencies = []

for i, query in enumerate(queries, start=1):
    start_time = time.perf_counter()

    response = query_engine.query(query)

    latency = time.perf_counter() - start_time
    original_latencies.append(latency)

    print(f"Query {i}: {latency:.4f} seconds")

average_original_latency = sum(original_latencies) / len(original_latencies)

print(f"\nAverage original index latency: {average_original_latency:.4f} seconds")

Query 1: 11.3683 seconds
Query 2: 6.9715 seconds
Query 3: 3.0033 seconds
Query 4: 8.3685 seconds
Query 5: 7.9474 seconds
Query 6: 8.1653 seconds
Query 7: 13.7601 seconds
Query 8: 7.6887 seconds
Query 9: 10.5902 seconds

Average original index latency: 8.6515 seconds


In [23]:
import time

query_10 = "What is the difference between supervised and unsupervised learning?"

start_time = time.perf_counter()
response_10_original = query_engine.query(query_10)
latency_10_original = time.perf_counter() - start_time

print("Query 10:", query_10)
print("Answer:", response_10_original)
print(f"Latency: {latency_10_original:.4f} seconds")

Query 10: What is the difference between supervised and unsupervised learning?
Answer: Supervised learning involves training models using labeled data, where each data point includes both inputs and the correct output. This method allows the model to learn the relationship between the inputs and outputs. In contrast, unsupervised learning works with unlabeled data, where the algorithm must find inherent patterns or groupings without predefined correct answers.
Latency: 14.6956 seconds


In [24]:
original_latencies_10 = [
    11.3683,
    6.9715,
    3.0033,
    8.3685,
    7.9474,
    8.1653,
    13.7601,
    7.6887,
    10.5902,
    14.6956
]

chroma_latencies_10 = [
    31.9751,
    13.2713,
    5.2812,
    13.8050,
    7.6182,
    10.8363,
    11.1386,
    16.2464,
    20.9139,
    14.8435
]

average_original = sum(original_latencies_10) / len(original_latencies_10)
average_chroma = sum(chroma_latencies_10) / len(chroma_latencies_10)

difference = average_chroma - average_original
percentage_change = (difference / average_original) * 100

print(f"Original VectorStoreIndex average: {average_original:.4f} seconds")
print(f"ChromaDB average: {average_chroma:.4f} seconds")
print(f"Difference: {difference:.4f} seconds")
print(f"Latency change: {percentage_change:.2f}%")

Original VectorStoreIndex average: 9.2559 seconds
ChromaDB average: 14.5930 seconds
Difference: 5.3371 seconds
Latency change: 57.66%
